In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(r"C:\Users\Acer\Desktop\projects\linear regression\car price prediction\quikr_car.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Cleaning

In [ ]:
df = df[df['year'].str.isnumeric()]

In [ ]:
df['year']=df['year'].astype(int)

In [ ]:
df.info()

In [ ]:
df = df[df['Price']!='Ask For Price']

In [ ]:
df['Price']= df['Price'].str.replace(',','').astype(int)

In [ ]:
df.info()

In [ ]:
df.drop('price',axis=1,inplace = True)

In [ ]:
df.info()

In [ ]:
df['kms_driven']=df['kms_driven'].str.split(' ').str.get(0).str.replace(',','')

In [ ]:
df = df[df['kms_driven'].str.isnumeric()]

In [ ]:
df['kms_driven'] = df['kms_driven'].astype(int)

In [ ]:
df.info()

In [ ]:
df = df[~df['fuel_type'].isna()]

In [ ]:
df['name']=df['name'].str.split(" ").str.slice(0,3).str.join(" ")

In [ ]:
df.reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df=df[df['Price']<6e6].reset_index(drop=True)

In [ ]:
df.to_csv('Cleaned_Car.csv')

## Model

In [ ]:
x=df.drop(columns='Price')
y=df['Price']

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [ ]:
ohe = OneHotEncoder()
ohe.fit(x[['name','company','fuel_type']])

In [ ]:
column_trans=make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),
                                    remainder='passthrough')

In [ ]:
lr=LinearRegression()

In [ ]:
pipe=make_pipeline(column_trans,lr)

In [ ]:
pipe.fit(x_train,y_train)

In [ ]:
y_pred=pipe.predict(x_test)

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
scores=[]
for i in range(1000):
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.1,random_state=i)
    lr=LinearRegression()
    pipe=make_pipeline(column_trans,lr)
    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)
    scores.append(r2_score(y_test,y_pred))

In [ ]:
np.argmax(scores)

In [ ]:
scores[np.argmax(scores)]

In [ ]:
!pip install pickle-mixin

In [ ]:
import pickle

In [ ]:
pickle.dump(pipe,open('LinearRegressionModel.pkl','wb'))

In [ ]:
pipe.predict(pd.DataFrame(columns=['name','company','year','kms_driven','fuel_type'],data=np.array(['Maruti Suzuki Swift','Maruti',2019,100,'Petrol']).reshape(1,5)))